In [2]:
import pandas as pd
from sqlalchemy import create_engine, text

In [3]:
# ─── CONFIGURAZIONE ───────────────────────────────────────────────
DB_USER     = "root"           # cambia con il tuo utente
DB_PASSWORD = "Matteo00"       # cambia con la tua password
DB_HOST     = "localhost"
DB_PORT     = 3306
DB_NAME     = "food_security"
CSV_PATH    = "global_food_security_intelligence.csv"
TABLE_NAME  = "food_data"
# ──────────────────────────────────────────────────────────────────

In [4]:
# 1. Connessione e creazione database se non esiste
engine_root = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}",
    echo=False
)
try:
    with engine_root.connect() as conn:
        conn.execute(text(f"CREATE DATABASE IF NOT EXISTS `{DB_NAME}` CHARACTER SET utf8mb4"))
        print(f"✓ Database '{DB_NAME}' pronto")
except Exception as e:
    print(f"Error: {e}")
    print("Please check your database credentials.")
 
# 2. Connessione al database specifico
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    echo=False
)
 

✓ Database 'food_security' pronto


In [5]:
print("Lettura CSV...")
df = pd.read_csv(CSV_PATH)
print(f"  → {len(df)} righe, {len(df.columns)} colonne")
 

Lettura CSV...
  → 5208 righe, 57 colonne


In [6]:
df.columns = [c.strip().replace(" ", "_").lower() for c in df.columns]

In [ ]:
# 5. Caricamento in MySQL
print(f"Caricamento in MySQL (tabella: {TABLE_NAME})...")
df.to_sql(
    name=TABLE_NAME,
    con=engine,
    if_exists="replace", 
    index=False,
    chunksize=500
)
print(f"✓ {len(df)} record caricati nella tabella '{TABLE_NAME}'")

Caricamento in MySQL (tabella: food_data)...
✓ 5208 record caricati nella tabella 'food_data'


In [10]:
VIEWS = {
 
    # ── PAGINA 1: OVERVIEW GLOBALE ────────────────────────────────
 
    "v01_kpi_globali": """
        SELECT
            COUNT(DISTINCT country_name)                        AS paesi_totali,
            MIN(year)                                           AS anno_inizio,
            MAX(year)                                           AS anno_fine,
            COUNT(*)                                            AS record_totali,
            SUM(food_crisis_flag)                               AS totale_anni_crisi,
            ROUND(AVG(hunger_severity_index), 2)                AS indice_fame_medio,
            ROUND(AVG(undernourishment_pct), 2)                 AS denutrizione_media_pct,
            ROUND(AVG(gdp_per_capita_usd), 0)                   AS pil_medio_usd,
            ROUND(AVG(child_mortality_per_1000), 1)             AS mortalita_infantile_media
        FROM food_data
    """,
 
    "v02_trend_annuale_globale": """
        SELECT
            year,
            ROUND(AVG(hunger_severity_index), 2)                AS indice_fame_medio,
            ROUND(AVG(undernourishment_pct), 2)                 AS denutrizione_pct,
            ROUND(AVG(gdp_per_capita_usd), 0)                   AS pil_pro_capite_medio,
            ROUND(AVG(child_mortality_per_1000), 1)             AS mortalita_infantile,
            ROUND(AVG(cereal_yield_kg_per_ha), 0)               AS resa_cereali_kg_ha,
            ROUND(AVG(food_production_index), 1)                AS indice_produzione_alimentare,
            SUM(food_crisis_flag)                               AS paesi_in_crisi,
            ROUND(SUM(food_crisis_flag) / COUNT(*) * 100, 1)   AS pct_paesi_in_crisi
        FROM food_data
        GROUP BY year
        ORDER BY year
    """,
 
    # ── PAGINA 2: MAPPA GEOGRAFICA ────────────────────────────────
 
    "v03_mappa_paesi_recente": """
        SELECT
            country_name,
            iso3,
            region,
            income_group,
            latitude,
            longitude,
            ROUND(AVG(hunger_severity_index), 1)                AS indice_fame_medio,
            ROUND(AVG(undernourishment_pct), 1)                 AS denutrizione_pct,
            ROUND(AVG(gdp_per_capita_usd), 0)                   AS pil_medio_usd,
            ROUND(AVG(child_mortality_per_1000), 1)             AS mortalita_infantile,
            SUM(food_crisis_flag)                               AS anni_in_crisi,
            COUNT(*)                                            AS anni_disponibili,
            ROUND(SUM(food_crisis_flag) / COUNT(*) * 100, 0)   AS pct_anni_crisi,
            CASE
                WHEN AVG(hunger_severity_index) < 5  THEN 'Basso'
                WHEN AVG(hunger_severity_index) < 15 THEN 'Moderato'
                WHEN AVG(hunger_severity_index) < 25 THEN 'Alto'
                ELSE 'Critico'
            END AS livello_rischio
        FROM food_data
        WHERE year BETWEEN 2019 AND 2023
        GROUP BY country_name, iso3, region, income_group, latitude, longitude
        HAVING COUNT(*) >= 2
        ORDER BY indice_fame_medio DESC
    """,
 
    "v04_analisi_per_regione": """
        SELECT
            region,
            COUNT(DISTINCT country_name)                        AS n_paesi,
            ROUND(AVG(hunger_severity_index), 2)                AS indice_fame_medio,
            ROUND(AVG(undernourishment_pct), 2)                 AS denutrizione_pct,
            ROUND(AVG(gdp_per_capita_usd), 0)                   AS pil_medio,
            ROUND(AVG(child_mortality_per_1000), 1)             AS mortalita_infantile,
            ROUND(AVG(cereal_yield_kg_per_ha), 0)               AS resa_cereali,
            ROUND(AVG(employment_in_agriculture_pct), 1)        AS lavoratori_agr_pct,
            SUM(food_crisis_flag)                               AS anni_in_crisi,
            ROUND(SUM(food_crisis_flag) / COUNT(*) * 100, 1)   AS pct_anni_crisi,
            ROUND(AVG(fao_basic_water_access_pct), 1)           AS accesso_acqua_pct,
            ROUND(AVG(fao_basic_sanitation_pct), 1)             AS igiene_base_pct
        FROM food_data
        GROUP BY region
        ORDER BY indice_fame_medio DESC
    """,
 
    # ── PAGINA 3: RANKING PAESI ───────────────────────────────────
 
    "v05_ranking_paesi": """
        SELECT
            country_name,
            iso3,
            region,
            income_group,
            ROUND(AVG(hunger_severity_index), 1)                AS indice_fame_medio,
            ROUND(AVG(undernourishment_pct), 1)                 AS denutrizione_pct,
            ROUND(AVG(gdp_per_capita_usd), 0)                   AS pil_medio_usd,
            ROUND(AVG(child_mortality_per_1000), 1)             AS mortalita_infantile,
            ROUND(AVG(employment_in_agriculture_pct), 1)        AS lavoratori_agr_pct,
            ROUND(AVG(cereal_yield_kg_per_ha), 0)               AS resa_cereali_kg_ha,
            SUM(food_crisis_flag)                               AS anni_in_crisi,
            COUNT(*)                                            AS anni_disponibili,
            CASE
                WHEN AVG(hunger_severity_index) < 5  THEN 'Basso'
                WHEN AVG(hunger_severity_index) < 15 THEN 'Moderato'
                WHEN AVG(hunger_severity_index) < 25 THEN 'Alto'
                ELSE 'Critico'
            END AS livello_rischio
        FROM food_data
        WHERE year BETWEEN 2019 AND 2023
        GROUP BY country_name, iso3, region, income_group
        HAVING COUNT(*) >= 2
        ORDER BY indice_fame_medio DESC
    """,
 
    # ── PAGINA 4: TREND PER PAESE ─────────────────────────────────
 
    "v06_serie_storica_per_paese": """
        SELECT
            country_name,
            iso3,
            region,
            income_group,
            year,
            ROUND(hunger_severity_index, 2)             AS indice_fame,
            ROUND(undernourishment_pct, 1)              AS denutrizione_pct,
            ROUND(gdp_per_capita_usd, 0)                AS pil_usd,
            ROUND(child_mortality_per_1000, 1)          AS mortalita_infantile,
            ROUND(cereal_yield_kg_per_ha, 0)            AS resa_cereali_kg_ha,
            ROUND(food_production_index, 1)             AS indice_produzione,
            ROUND(inflation_consumer_prices_pct, 1)     AS inflazione_pct,
            ROUND(employment_in_agriculture_pct, 1)     AS lavoratori_agr_pct,
            ROUND(fao_dietary_energy_adequacy_pct, 1)   AS adeguatezza_energetica_pct,
            food_crisis_flag,
            row_completeness_pct
        FROM food_data
        ORDER BY country_name, year
    """,
 
    "v07_variazione_storica_paese": """
        SELECT
            t1.country_name,
            t1.region,
            t1.income_group,
            ROUND(t1.fame_inizio, 1)                            AS indice_fame_2001_05,
            ROUND(t2.fame_fine, 1)                              AS indice_fame_2019_23,
            ROUND(t2.fame_fine - t1.fame_inizio, 1)             AS variazione_assoluta,
            ROUND((t2.fame_fine - t1.fame_inizio)
                  / NULLIF(t1.fame_inizio, 0) * 100, 1)         AS variazione_pct,
            CASE
                WHEN t2.fame_fine < t1.fame_inizio THEN 'Migliorato'
                WHEN t2.fame_fine > t1.fame_inizio THEN 'Peggiorato'
                ELSE 'Stabile'
            END AS tendenza
        FROM (
            SELECT country_name, region, income_group,
                   AVG(hunger_severity_index) AS fame_inizio
            FROM food_data
            WHERE year BETWEEN 2001 AND 2005
            GROUP BY country_name, region, income_group
        ) t1
        JOIN (
            SELECT country_name,
                   AVG(hunger_severity_index) AS fame_fine
            FROM food_data
            WHERE year BETWEEN 2019 AND 2023
            GROUP BY country_name
        ) t2 ON t1.country_name = t2.country_name
        ORDER BY variazione_assoluta ASC
    """,
 
    # ── PAGINA 5: CORRELAZIONI E FATTORI ─────────────────────────
 
    "v08_scatter_fattori_fame": """
        SELECT
            country_name,
            region,
            income_group,
            year,
            ROUND(hunger_severity_index, 1)             AS indice_fame,
            ROUND(gdp_per_capita_usd, 0)                AS pil_usd,
            ROUND(employment_in_agriculture_pct, 1)     AS lavoratori_agr_pct,
            ROUND(child_mortality_per_1000, 1)          AS mortalita_infantile,
            ROUND(cereal_yield_kg_per_ha, 0)            AS resa_cereali_kg_ha,
            ROUND(fao_basic_water_access_pct, 1)        AS accesso_acqua_pct,
            ROUND(fao_basic_sanitation_pct, 1)          AS igiene_base_pct,
            ROUND(undernourishment_pct, 1)              AS denutrizione_pct,
            ROUND(inflation_consumer_prices_pct, 1)     AS inflazione_pct,
            food_crisis_flag
        FROM food_data
        WHERE hunger_severity_index IS NOT NULL
          AND gdp_per_capita_usd IS NOT NULL
        ORDER BY year, country_name
    """,
 
    "v09_analisi_reddito_nel_tempo": """
        SELECT
            income_group,
            year,
            COUNT(DISTINCT country_name)                        AS n_paesi,
            ROUND(AVG(hunger_severity_index), 2)                AS indice_fame_medio,
            ROUND(AVG(undernourishment_pct), 2)                 AS denutrizione_pct,
            ROUND(AVG(gdp_per_capita_usd), 0)                   AS pil_medio_usd,
            ROUND(AVG(child_mortality_per_1000), 1)             AS mortalita_infantile,
            ROUND(AVG(employment_in_agriculture_pct), 1)        AS lavoratori_agr_pct,
            ROUND(AVG(cereal_yield_kg_per_ha), 0)               AS resa_cereali,
            ROUND(AVG(fao_basic_water_access_pct), 1)           AS accesso_acqua_pct,
            SUM(food_crisis_flag)                               AS anni_crisi
        FROM food_data
        GROUP BY income_group, year
        ORDER BY income_group, year
    """,
 
    # ── PAGINA 6: CRISI E SHOCK ───────────────────────────────────
 
    "v10_impatto_covid": """
        SELECT
            region,
            income_group,
            ROUND(AVG(CASE WHEN year = 2019 THEN hunger_severity_index END), 2)         AS fame_2019,
            ROUND(AVG(CASE WHEN year = 2020 THEN hunger_severity_index END), 2)         AS fame_2020,
            ROUND(AVG(CASE WHEN year = 2021 THEN hunger_severity_index END), 2)         AS fame_2021,
            ROUND(AVG(CASE WHEN year = 2022 THEN hunger_severity_index END), 2)         AS fame_2022,
            ROUND(AVG(CASE WHEN year IN (2020,2021) THEN hunger_severity_index END)
                - AVG(CASE WHEN year = 2019 THEN hunger_severity_index END), 2)         AS impatto_covid,
            ROUND(AVG(CASE WHEN year = 2019 THEN undernourishment_pct END), 2)          AS denutr_2019,
            ROUND(AVG(CASE WHEN year IN (2020,2021) THEN undernourishment_pct END), 2)  AS denutr_covid
        FROM food_data
        GROUP BY region, income_group
        ORDER BY impatto_covid DESC
    """,
 
    "v11_paesi_crisi_cronica": """
        SELECT
            country_name,
            region,
            income_group,
            SUM(food_crisis_flag)                               AS anni_in_crisi,
            COUNT(*)                                            AS anni_totali,
            ROUND(SUM(food_crisis_flag) / COUNT(*) * 100, 0)   AS pct_anni_crisi,
            ROUND(AVG(hunger_severity_index), 1)                AS indice_fame_medio,
            ROUND(AVG(gdp_per_capita_usd), 0)                   AS pil_medio_usd,
            MIN(year)                                           AS primo_anno_crisi,
            MAX(year)                                           AS ultimo_anno_crisi
        FROM food_data
        GROUP BY country_name, region, income_group
        HAVING SUM(food_crisis_flag) >= 10
        ORDER BY anni_in_crisi DESC
    """,
 
    "v12_crisi_timeline_regione": """
        SELECT
            year,
            region,
            COUNT(DISTINCT CASE WHEN food_crisis_flag = 1 THEN country_name END)        AS paesi_in_crisi,
            COUNT(DISTINCT country_name)                                                 AS paesi_totali,
            ROUND(AVG(CASE WHEN food_crisis_flag = 1
                      THEN hunger_severity_index END), 1)                                AS fame_media_crisi
        FROM food_data
        GROUP BY year, region
        ORDER BY year, region
    """,
 
    # ── VISTA PRINCIPALE COMPLETA PER POWER BI ────────────────────
 
    "v00_export_powerbi": """
        SELECT
            country_name,
            iso3,
            year,
            region,
            income_group,
            latitude,
            longitude,
            ROUND(hunger_severity_index, 2)             AS hunger_severity_index,
            ROUND(undernourishment_pct, 2)              AS undernourishment_pct,
            ROUND(gdp_per_capita_usd, 0)                AS gdp_per_capita_usd,
            ROUND(child_mortality_per_1000, 1)          AS child_mortality_per_1000,
            ROUND(cereal_yield_kg_per_ha, 0)            AS cereal_yield_kg_per_ha,
            ROUND(employment_in_agriculture_pct, 1)     AS employment_in_agriculture_pct,
            ROUND(food_production_index, 1)             AS food_production_index,
            ROUND(fao_dietary_energy_adequacy_pct, 1)   AS dietary_energy_adequacy_pct,
            ROUND(fao_basic_water_access_pct, 1)        AS basic_water_access_pct,
            ROUND(fao_basic_sanitation_pct, 1)          AS basic_sanitation_pct,
            ROUND(inflation_consumer_prices_pct, 1)     AS inflation_pct,
            ROUND(fao_protein_supply_g_per_day, 1)      AS protein_supply_g_day,
            ROUND(food_crop_production_gap, 1)          AS food_crop_production_gap,
            food_crisis_flag,
            CASE
                WHEN hunger_severity_index < 5  THEN 'Basso'
                WHEN hunger_severity_index < 15 THEN 'Moderato'
                WHEN hunger_severity_index < 25 THEN 'Alto'
                ELSE 'Critico'
            END AS livello_rischio,
            row_completeness_pct
        FROM food_data
        ORDER BY country_name, year
    """
}
 
# ─── CREAZIONE DELLE VIEW ─────────────────────────────────────────
 
def create_all_views():
    created, failed = [], []
 
    with engine.connect() as conn:
        for view_name, select_sql in VIEWS.items():
            try:
                # DROP + CREATE per aggiornare view esistenti
                conn.execute(text(f"DROP VIEW IF EXISTS `{view_name}`"))
                conn.execute(text(
                    f"CREATE VIEW `{view_name}` AS {select_sql}"
                ))
                conn.commit()
                print(f"  ✓ {view_name}")
                created.append(view_name)
            except Exception as e:
                print(f"  ✗ {view_name} — ERRORE: {e}")
                failed.append(view_name)
 
    print(f"\n{'='*50}")
    print(f"  Create:  {len(created)}/{len(VIEWS)} view")
    if failed:
        print(f"  Fallite: {failed}")
    print(f"{'='*50}")
    return created, failed
 
 
def list_views():
    """Mostra tutte le view presenti nel DB con il numero di righe."""
    with engine.connect() as conn:
        views = conn.execute(text(
            f"SELECT TABLE_NAME FROM information_schema.VIEWS "
            f"WHERE TABLE_SCHEMA = '{DB_NAME}' ORDER BY TABLE_NAME"
        )).fetchall()
 
        print(f"\nView presenti in '{DB_NAME}':")
        for (v,) in views:
            try:
                n = conn.execute(text(f"SELECT COUNT(*) FROM `{v}`")).scalar()
                print(f"  {v:<40} {n:>6} righe")
            except Exception as e:
                print(f"  {v:<40} errore: {e}")
 
 
def drop_all_views():
    """Rimuove tutte le view (utile per reset completo)."""
    with engine.connect() as conn:
        for view_name in VIEWS:
            conn.execute(text(f"DROP VIEW IF EXISTS `{view_name}`"))
            conn.commit()
            print(f"  Rimossa: {view_name}")
    print("✓ Tutte le view eliminate.")
 
 
if __name__ == "__main__":
    print(f"\nCreazione di {len(VIEWS)} view nel database '{DB_NAME}'...\n")
    create_all_views()
    list_views()
    print("\n✓ Fatto! In Power BI connettiti a MySQL e seleziona le view v00–v12.")


Creazione di 13 view nel database 'food_security'...

  ✓ v01_kpi_globali
  ✓ v02_trend_annuale_globale
  ✓ v03_mappa_paesi_recente
  ✓ v04_analisi_per_regione
  ✓ v05_ranking_paesi
  ✓ v06_serie_storica_per_paese
  ✓ v07_variazione_storica_paese
  ✓ v08_scatter_fattori_fame
  ✓ v09_analisi_reddito_nel_tempo
  ✓ v10_impatto_covid
  ✓ v11_paesi_crisi_cronica
  ✓ v12_crisi_timeline_regione
  ✓ v00_export_powerbi

  Create:  13/13 view

View presenti in 'food_security':
  v00_export_powerbi                         5208 righe
  v01_kpi_globali                               1 righe
  v02_trend_annuale_globale                    24 righe
  v03_mappa_paesi_recente                     217 righe
  v04_analisi_per_regione                       7 righe
  v05_ranking_paesi                           217 righe
  v06_serie_storica_per_paese                5208 righe
  v07_variazione_storica_paese                217 righe
  v08_scatter_fattori_fame                   5041 righe
  v09_analisi_reddito_n